# ScholarshipCoach — Pipeline Walkthrough

A step-by-step demonstration of the full scholarship recommendation pipeline:
**Stage 0** (data) → **Stage 1** (eligibility) → **Stage 2** (scoring) → **Stage 3** (reranking).

The profile used throughout is **NC, Computer Science, rising sophomore, GPA 3.25**.

In [ ]:
import sys
from pathlib import Path

_root = Path.cwd()
if _root.name == "notebooks":
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from datetime import date
from collections import Counter

plt.style.use("dark_background")
plt.rcParams.update({
    "figure.dpi": 100,
    "figure.figsize": (10, 4),
    "axes.spines.top": False,
    "axes.spines.right": False,
})
pd.set_option("display.max_colwidth", 60)

In [ ]:
try:
    from src.win_model.infer import load_latest_model
    _win_model = load_latest_model()
    HAS_WIN_MODEL = True
    print("Win model loaded.")
except FileNotFoundError:
    _win_model = None
    HAS_WIN_MODEL = False
    print("No win model found — Sections 4 & 5 will run without it.")

## 1. Data Overview

The catalog is stored as a Parquet snapshot. We load the latest file and inspect its shape, key columns, and the distribution of award amounts and deadlines.

In [ ]:
snapshot_dir = _root / "data" / "processed"
snapshots = sorted(snapshot_dir.glob("scholarships_snapshot_*.parquet"))
snapshot_path = snapshots[-1]
print(f"Snapshot: {snapshot_path.name}")

df = pd.read_parquet(snapshot_path)
print(f"Shape:   {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")

In [ ]:
fig, ax = plt.subplots()
amounts = df["amount_max"].dropna()
ax.hist(amounts.clip(upper=50_000), bins=40, color="#4C9BE8", edgecolor="none")
ax.set_xlabel("Award Amount ($)")
ax.set_ylabel("Count")
ax.set_title("Distribution of Award Amounts (capped at $50k)")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
plt.tight_layout()
plt.show()
print(f"Median: ${amounts.median():,.0f}  |  Max: ${amounts.max():,.0f}")

In [ ]:
df["deadline_dt"] = pd.to_datetime(df["deadline"], errors="coerce")
monthly = df["deadline_dt"].dt.to_period("M").value_counts().sort_index().head(24)

fig, ax = plt.subplots()
ax.bar(monthly.index.astype(str), monthly.values, color="#4CE89B", edgecolor="none")
ax.set_xlabel("Month")
ax.set_ylabel("Scholarships")
ax.set_title("Scholarship Deadlines by Month")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
cols = ["title", "amount_max", "deadline", "states_allowed", "sponsor"]
df[cols].dropna(subset=["amount_max"]).head(5)

> **Key insight:** The catalog spans many states and award tiers, with median award around $10k. Deadlines cluster seasonally, giving the urgency-boost signal meaningful leverage.

## 2. Stage 1 — Eligibility Filtering

Stage 1 applies hard constraints (GPA, state, major, education level, citizenship, deadline) and annotates each ineligible scholarship with reason codes so we can understand the filter behaviour.

In [ ]:
from src.rank.stage1_eligibility import StudentProfile, apply_eligibility_filter

NC_TODAY = date(2026, 7, 1)
nc_stage1 = StudentProfile(
    gpa=3.25, state="NC", major="Computer Science",
    education_level="high school", citizenship="US", today=NC_TODAY,
)

eligible_df, ineligible_df = apply_eligibility_filter(df, nc_stage1)
print(f"Total: {len(df):,}  |  Eligible: {len(eligible_df):,}  |  Ineligible: {len(ineligible_df):,}")

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(["Eligible", "Ineligible"], [len(eligible_df), len(ineligible_df)],
       color=["#4CE89B", "#E84C4C"])
ax.set_ylabel("Count")
ax.set_title("Stage 1 Eligibility Split")
plt.tight_layout()
plt.show()

In [ ]:
all_reasons = [r for row in ineligible_df["reasons"] for r in row]
reason_counts = Counter(all_reasons)

fig, ax = plt.subplots(figsize=(8, 3))
ax.barh(list(reason_counts.keys()), list(reason_counts.values()), color="#E8A84C")
ax.set_xlabel("Count")
ax.set_title("Ineligibility Reason Codes")
plt.tight_layout()
plt.show()

In [ ]:
ineligible_df[["title", "amount_max", "reasons"]].head(8)

> **Key insight:** Most rejections are DEADLINE_PASSED and STATE_NOT_ALLOWED — structural constraints, not profile-specific. The education-level filter is intentionally strict for a high-school student profile.

## 3. Stage 2 — Scoring Components

Stage 2 scores each eligible scholarship on four components, then combines them with configurable weights. We use the Pareto-selected weights from the weight-tuning run.

In [ ]:
from src.rank.stage2_scoring import score_stage2
from src.rank.weights import Stage2Weights

NC_PROFILE_STAGE2 = {
    "major": "Computer Science",
    "interests": ["programming", "robotics", "game development", "cybersecurity", "math"],
    "keywords": ["STEM", "computer science", "engineering", "technology", "coding", "software"],
    "extracurriculars": ["robotics club", "math team", "coding bootcamp", "volunteer tutoring"],
    "goals": "Pursuing a degree in CS/CE with interest in software development and cybersecurity",
}

pareto_w2 = Stage2Weights(text_sim=0.70, amount=0.10, keyword=0.15, effort=0.05)
scored_df = score_stage2(eligible_df, NC_PROFILE_STAGE2, weights=pareto_w2)
print(f"Weights: {pareto_w2}")
scored_df[["title", "text_sim", "amount_utility", "keyword_overlap", "effort_penalty", "stage2_score"]].head(5)

In [ ]:
top10_s2 = scored_df.nlargest(10, "stage2_score").reset_index(drop=True)
labels = [t[:35] for t in top10_s2["title"]]
components = ["text_sim", "amount_utility", "keyword_overlap"]
colours = ["#4C9BE8", "#4CE89B", "#E8D04C"]

fig, ax = plt.subplots(figsize=(11, 4))
bottom = np.zeros(10)
for col, colour in zip(components, colours):
    ax.bar(range(10), top10_s2[col], bottom=bottom, label=col, color=colour)
    bottom += top10_s2[col].values
ax.set_xticks(range(10))
ax.set_xticklabels(labels, rotation=40, ha="right", fontsize=8)
ax.set_ylabel("Component Score")
ax.set_title("Stage 2 Component Contributions — Top 10")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
sc = ax.scatter(
    scored_df["text_sim"], scored_df["amount_utility"],
    c=scored_df["stage2_score"], cmap="viridis", alpha=0.7, s=40,
)
plt.colorbar(sc, ax=ax, label="stage2_score")
ax.set_xlabel("text_sim")
ax.set_ylabel("amount_utility")
ax.set_title("Text Similarity vs Amount Utility (coloured by stage2_score)")
plt.tight_layout()
plt.show()

> **Key insight:** `text_sim` dominates the composite score (70% weight). Scholarships with high amount utility but low text similarity still rank below well-matched ones — this is the intended behaviour: relevance before value.

## 4. Stage 3 — Decision Reranking

Stage 3 adds an urgency boost (exponential decay over days-to-deadline) and an expected-value signal on top of the Stage 2 score. When the win model is available it estimates P(award) to sharpen EV.

In [ ]:
days = np.linspace(0, 90, 300)
boost = np.exp(-days / 30.0)

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(days, boost, color="#E84C9B", lw=2)
ax.set_xlabel("Days to Deadline")
ax.set_ylabel("Urgency Boost")
ax.set_title("Urgency Boost Curve: exp(−days / 30)")
ax.axvline(30, ls="--", color="gray", alpha=0.6, label="30 days → boost = 0.37")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
from src.rank.stage3_rerank import rerank_stage3
from src.rank.weights import Stage3Weights

pareto_w3 = Stage3Weights(stage2=0.90, urgency=0.05, ev=0.05)

reranked_df = rerank_stage3(
    scored_df, NC_TODAY, profile=NC_PROFILE_STAGE2,
    weights=pareto_w3, use_win_model=HAS_WIN_MODEL, win_model=_win_model,
)

cols = ["title", "stage2_score", "urgency_boost", "final_score"]
if HAS_WIN_MODEL:
    cols += ["p_win", "expected_value"]
reranked_df[cols].head(10)

In [ ]:
# Stage 2 rank vs Stage 3 final rank
s2_order = scored_df.sort_values("stage2_score", ascending=False).reset_index(drop=True)
s2_order["stage2_rank"] = s2_order.index + 1

merged = reranked_df.head(10).merge(
    s2_order[["scholarship_id", "stage2_rank"]], on="scholarship_id", how="left"
)
merged["final_rank"] = range(1, len(merged) + 1)
merged["rank_delta"] = merged["stage2_rank"] - merged["final_rank"]

fig, ax = plt.subplots(figsize=(11, 3))
bar_colours = ["#4CE89B" if d > 0 else "#E84C4C" if d < 0 else "gray"
               for d in merged["rank_delta"]]
ax.bar(range(len(merged)), merged["rank_delta"], color=bar_colours)
ax.set_xticks(range(len(merged)))
ax.set_xticklabels([t[:30] for t in merged["title"]], rotation=40, ha="right", fontsize=8)
ax.axhline(0, color="white", lw=0.5)
ax.set_ylabel("Stage 2 rank − Final rank (↑ = moved up)")
ax.set_title("Stage 3 Rank Shifts vs Stage 2")
plt.tight_layout()
plt.show()

> **Key insight:** Urgency reorders scholarships with imminent deadlines above those with equal Stage 2 scores but later deadlines. The EV signal (when the win model is active) further prioritises high-value, winnable opportunities.

## 5. Win Probability Model

The win model estimates P(award) from scholarship features — amount, deadline distance, keyword overlap, and text similarity — providing an expected-value signal: **EV = P(win) × award amount**.

In [ ]:
if not HAS_WIN_MODEL:
    print("Win model not available. Skipping Section 5.")
else:
    from src.win_model.features import FEATURE_COLUMNS
    try:
        estimator = _win_model.model[-1]
        if hasattr(estimator, "coef_"):
            importances = pd.Series(
                np.abs(estimator.coef_[0]), index=FEATURE_COLUMNS
            ).sort_values(ascending=True)
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.barh(importances.index, importances.values, color="#9B4CE8")
            ax.set_title("Win Model — Feature Importances (|coef|)")
            ax.set_xlabel("|Coefficient|")
            plt.tight_layout()
            plt.show()
        elif hasattr(estimator, "feature_importances_"):
            importances = pd.Series(
                estimator.feature_importances_, index=FEATURE_COLUMNS
            ).sort_values(ascending=True)
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.barh(importances.index, importances.values, color="#9B4CE8")
            ax.set_title("Win Model — Feature Importances")
            plt.tight_layout()
            plt.show()
        else:
            print(f"Model type {type(estimator).__name__} — importances not directly accessible.")
    except Exception as exc:
        print(f"Could not extract feature importances: {exc}")

In [ ]:
if HAS_WIN_MODEL and "p_win" in reranked_df.columns:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.scatter(
        reranked_df["amount_max"].clip(upper=50_000),
        reranked_df["p_win"],
        c=reranked_df["stage2_score"], cmap="plasma", alpha=0.7, s=40,
    )
    ax1.set_xlabel("Award Amount ($, capped at 50k)")
    ax1.set_ylabel("P(Win)")
    ax1.set_title("P(Win) vs Award Amount")
    ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))

    ax2.hist(reranked_df["p_win"].dropna(), bins=20, color="#9B4CE8", edgecolor="none")
    ax2.set_xlabel("P(Win)")
    ax2.set_ylabel("Count")
    ax2.set_title("P(Win) Distribution")

    plt.tight_layout()
    plt.show()
    print(f"Median P(Win): {reranked_df['p_win'].median():.3f}")
elif HAS_WIN_MODEL:
    print("Re-run Stage 3 with use_win_model=True to populate p_win column.")

> **Key insight:** The win model correctly assigns higher P(win) to scholarships with closer profile matches. High-amount scholarships with poor profile fit receive low P(win), dampening their EV score so they don't crowd out better-matched awards.

## 6. Weight Tuning & Pareto Front

We swept 200 weight configurations and selected the Pareto-optimal set — configurations that simultaneously maximise NDCG (ranking quality) and coverage (diversity of sponsors/sources).

In [ ]:
artifact_dir = _root / "reports" / "weight_tuning" / "artifacts"
artifact_files = sorted(artifact_dir.glob("weight_tuning_*.json"))
latest_artifact = json.loads(artifact_files[-1].read_text(encoding="utf-8"))

configs = latest_artifact["all_configs"]
pareto  = latest_artifact["pareto_front"]
print(f"Artifact : {artifact_files[-1].name}")
print(f"All configs : {len(configs)}, Pareto front : {len(pareto)}")

In [ ]:
ndcg_all = [c["metrics"]["ndcg_at_k"] for c in configs]
cov_all  = [c["metrics"]["coverage_at_k"] for c in configs]

ndcg_pf = [c["metrics"]["ndcg_at_k"] for c in pareto]
cov_pf  = [c["metrics"]["coverage_at_k"] for c in pareto]

knee_idx = max(range(len(pareto)), key=lambda i: pareto[i].get("pareto_knee_score", 0))

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(cov_all, ndcg_all, alpha=0.3, s=20, color="gray", label="All configs")
ax.scatter(cov_pf,  ndcg_pf,  s=60,  color="#4CE89B", zorder=5, label="Pareto front")
ax.scatter(
    [cov_pf[knee_idx]], [ndcg_pf[knee_idx]], s=200, color="#E84C4C",
    zorder=6, marker="*", label="Selected knee",
)
ax.set_xlabel("Coverage @ k=10")
ax.set_ylabel("NDCG @ k=10")
ax.set_title("Weight Tuning — Pareto Front (NDCG vs Coverage)")
ax.legend()
plt.tight_layout()
plt.show()

knee = pareto[knee_idx]
print("Selected knee config:")
print(f"  Stage2: {knee['stage2_weights']}")
print(f"  Stage3: {knee['stage3_weights']}")
print(f"  NDCG={ndcg_pf[knee_idx]:.4f}, Coverage={cov_pf[knee_idx]:.4f}")

> **Key insight:** Pareto selection identifies configurations that don't sacrifice coverage for ranking quality or vice versa. The selected knee point achieves strong NDCG while maintaining meaningful sponsor and source diversity.

## 7. Full Pipeline for Your Student

Running the complete pipeline end-to-end for the NC, CS, rising-sophomore student profile to produce a personalised, urgency-aware top-10 recommendation list.

In [ ]:
# Stage 1
eligible7, ineligible7 = apply_eligibility_filter(df, nc_stage1)

# Stage 2
scored7 = score_stage2(eligible7, NC_PROFILE_STAGE2, weights=pareto_w2)

# Stage 3
reranked7 = rerank_stage3(
    scored7, NC_TODAY, profile=NC_PROFILE_STAGE2,
    weights=pareto_w3, use_win_model=HAS_WIN_MODEL, win_model=_win_model,
)

print(f"Stage 1: {len(eligible7)} eligible / {len(ineligible7)} filtered")
print(f"Stage 2: {len(scored7)} scored")
print(f"Stage 3: {len(reranked7)} reranked")

In [ ]:
display_cols = ["title", "sponsor", "amount_max", "deadline", "stage2_score", "urgency_boost", "final_score"]
if HAS_WIN_MODEL:
    display_cols += ["p_win", "expected_value"]

top10_final = reranked7[display_cols].head(10).copy()
top10_final["amount_max"] = top10_final["amount_max"].apply(
    lambda x: f"${x:,.0f}" if pd.notna(x) else "—"
)
top10_final["deadline"] = pd.to_datetime(
    top10_final["deadline"], errors="coerce"
).dt.strftime("%Y-%m-%d")
top10_final.index = range(1, len(top10_final) + 1)
top10_final.index.name = "Rank"
top10_final

> **Key insight:** The pipeline distils hundreds of raw scholarships to a personalised, urgency-aware top-10 list. Expected value combines win probability and award size to surface high-ROI opportunities that pure text similarity would miss.